In [ ]:
import json
from pyspark.sql import functions as F
from notebookutils import mssparkutils
WS_ID     = "52542446-5d75-45b5-b949-2a9acd365317"
BRONZE_ID = "3beed14c-c291-4bcf-aafd-cbfe46a65d89"
tbl = f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}/Tables/pp_telemetry_raw"
out = f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}/Files/inspect.json"
df = spark.read.format('delta').load(tbl)
schema_str = df.schema.json()
counts = {r.eventType: r['count'] for r in df.groupBy('eventType').count().collect()}
samples = {}
for et in counts.keys():
    rows = df.filter(F.col('eventType') == et).limit(2).collect()
    samples[et] = []
    for r in rows:
        d = {}
        for k, v in r.asDict(recursive=True).items():
            try:
                json.dumps(v, default=str)
                d[k] = v
            except Exception:
                d[k] = str(v)
        samples[et].append(d)
result = {'schema': schema_str, 'counts': counts, 'samples': samples, 'total': df.count()}
tmp = out + '.tmp'
sc.parallelize([json.dumps(result, default=str, indent=2)]).coalesce(1).saveAsTextFile(tmp)
files = mssparkutils.fs.ls(tmp)
part = [f for f in files if f.name.startswith('part-')][0]
mssparkutils.fs.cp(part.path, out, True)
mssparkutils.fs.rm(tmp, True)
print('wrote', out)